# 第 3 章:配置 —— 每个超参数为何这样选

在上一章中,我们学会了用 tokenizer 把文本切成 token id。但 token id 进入模型后,究竟要经过一个**多大**的模型?有几层?每层多宽?多少个注意力头?

这些问题全部由一个 ~50 行的类来回答:**`MiniMindConfig`**。

本章逐行拆解 `MiniMindConfig — model_minimind.py:~10-50 (@67f114a)` 的每一个超参数,解释**为什么是这个值**,并用代码验证每一步的参数量计算。读完本章,你将完全理解 minimind「64M 参数」这个数字是怎么来的。

> 🔧 **Fork 扩展 (可选阅读)**: `use_ple` / `ple_dim` (行 34–35) 是本 fork 为 ESP32-S3 边缘部署新增的 PLE (Per-Layer Embedding) 稀疏每层嵌入表配置。**默认 `False`,完全不执行任何 PLE 逻辑**,模型行为与上游 minimind 原版 100% 一致。如需了解 PLE 架构细节,参考 esp32-ai 仓库。本教程后续章节默认 `use_ple=False`。

## 3.1 为什么先讲配置

回顾第 1 章的推理代码:

```python
config = MiniMindConfig(hidden_size=768, num_hidden_layers=8)
model = MiniMindForCausalLM(config)
```

**`config` 是模型的「身份证」。** 它定义了模型的一切结构信息 —— 维度、层数、头数、激活函数、词表大小。模型代码里的每一个 `nn.Linear`、`nn.Embedding` 的尺寸,都是从 config 里读出来的。

后续 4 章(注意力、FFN、模型组装、生成)的每一行代码都会引用 config 的字段。所以我们必须先把这张「身份证」的每个字段弄清楚。

> 这和造房子一样:config 是**蓝图**,模型是**房子**。不读蓝图就盖房,只能盲人摸象。

In [ ]:
# 先看完整的 config 定义(MiniMindConfig — model_minimind.py:~10-50 @67f114a)
# 我们逐行拆解,先建立全貌

config_fields = {
    "hidden_size":            ("768",   "隐藏维度 d_model,所有残差流的宽度"),
    "num_hidden_layers":      ("8",     "Transformer Block 的层数(深度)"),
    "vocab_size":             ("6400",  "词表大小(第 2 章已讲)"),
    "num_attention_heads":    ("8",     "Query 头数"),
    "num_key_value_heads":    ("4",     "Key/Value 头数(GQA,比 Q 少)"),
    "head_dim":               ("96",    "每个头的维度 = 768 // 8"),
    "intermediate_size":      ("2432",  "FFN 中间层宽度 = ceil(768·pi/64)·64"),
    "hidden_act":             ("'silu'", "SwiGLU 的激活函数"),
    "max_position_embeddings":("32768", "最大序列长度"),
    "rope_theta":             ("1e6",   "RoPE 的基础频率"),
    "tie_word_embeddings":    ("True",  "输入嵌入与输出层共享权重"),
    "rms_norm_eps":           ("1e-6",  "RMSNorm 的 epsilon"),
    "dropout":                ("0.0",   "推理时关闭,训练时可设非零"),
    "use_ple":                ("False", "🔧 Fork 扩展:PLE 每层嵌入(行~34),ESP32 部署用,默认关闭不影响行为"),
    "ple_dim":                ("hidden_size//4", "🔧 Fork 扩展:PLE 嵌入维度(行~35)"),
    "use_moe":                ("False", "是否使用 MoE"),
    "num_experts":            ("4",     "MoE 专家数(use_moe=True 时生效)"),
    "num_experts_per_tok":    ("1",     "每个 token 激活的专家数(top-1)"),
    "router_aux_loss_coef":   ("5e-4",  "MoE 负载均衡 loss 系数"),
}

for name, (val, desc) in config_fields.items():
    print(f"  {name:30s} = {val:8s}  # {desc}")

这 18 个字段可以分成 7 组,后续章节逐组拆解:

| 组 | 字段 | 涉及章节 |
|---|---|---|
| 核心维度 | `hidden_size`, `num_hidden_layers` | **本章 3.2** |
| 注意力 | `num_attention_heads`, `num_key_value_heads`, `head_dim` | **本章 3.3** + 第 4 章 |
| FFN | `intermediate_size`, `hidden_act` | **本章 3.4** + 第 5 章 |
| 位置编码 | `rope_theta`, `max_position_embeddings`, `rope_scaling` | **本章 3.5** + 第 4 章 |
| 词表与绑定 | `vocab_size`, `tie_word_embeddings` | **本章 3.6** |
| 🔧 PLE | `use_ple`, `ple_dim` | 🔧 Fork 扩展(默认关闭,不影响行为) |
| MoE | `use_moe`, `num_experts`, `num_experts_per_tok`, `router_aux_loss_coef` | **本章 3.7** + 第 15 章 |

> 其余字段(`dropout`, `rms_norm_eps`, `flash_attn`)是训练/推理时的工程细节,不影响参数量,会在用到时解释。

&nbsp;

---

## 3.2 核心维度:hidden_size 与 num_hidden_layers

### 3.2.1 hidden_size = 768(宽度)

`hidden_size` 也叫 `d_model`,是 token embedding 的维度,也是所有 Transformer Block 内部残差流的宽度。每个 token 在模型里始终是一个 768 维向量,直到最后一层被 lm_head 映射回词表维度。

**为什么是 768?**

768 不是一个随意的数字 —— 它恰好是 BERT-base / GPT-2 small / ViT-base 的隐藏维度,是 NLP 领域最经典的「小模型维度」之一。minimind 选择 768 的核心考量是:

| 维度 | 参数量 | 关键问题 |
|---|---|---|
| 256 | ~7M | 表示瓶颈:MobileLLM 论文(Meta, 2024)表明,dim < 512 时模型严重欠拟合 |
| 512 | ~27M | 可用但偏小,在多语言/代码任务上泛化差 |
| **768** | **~64M** | **甜点:参数效率和表示能力的平衡点** |
| 1024 | ~114M | 好但训练成本翻倍,偏离「¥3 训练」目标 |

> **MobileLLM 的启示**:Meta 在 MobileLLM 论文中系统研究了 < 350M 参数的小模型,发现 `hidden_size < 500` 时模型存在严重的「表示瓶颈」—— 即使层数再多,每层的表示能力也不足以捕捉复杂的语言模式。768 足够宽,能避开这个瓶颈。

### 3.2.2 num_hidden_layers = 8(深度)

层数决定模型能做多深的「组合推理」。每一层 Transformer Block 都是一次 attention + FFN 的变换,层与层之间通过残差连接堆叠。

**为什么是 8 层?**

经验法则:小模型的有效深度通常在 6-12 层。GPT-2 small 是 12 层 / 768 维,但参数量 124M。minimind 只用 8 层,搭配 768 维,正好落在 ~64M 的参数预算内。

8 层的另一个好处:**KV cache 更小**。每层都会缓存 Key/Value,层数越少,推理时的显存占用越低(第 7 章详解)。

In [ ]:
import math

hidden_size = 768
num_layers = 8
vocab_size = 6400

# 粗估:每层参数 ≈ 12 × d²(经验公式,粗略)
per_layer_rough = 12 * hidden_size ** 2
total_rough = vocab_size * hidden_size + num_layers * per_layer_rough
print(f"粗估每层参数(12·d²):  {per_layer_rough:>12,}")
print(f"粗估总参数:            {total_rough:>12,} ≈ {total_rough/1e6:.0f}M")
print(f"(粗估值偏高,精确计算在 3.8 节)")
print()

# 对比不同配置
configs = [
    ("dim=256, L=8",  256, 8),
    ("dim=512, L=6",  512, 6),
    ("dim=768, L=8  ← minimind", 768, 8),
    ("dim=768, L=12 (GPT-2 small)", 768, 12),
    ("dim=1024, L=8", 1024, 8),
]
print(f"{'配置':<30s} {'~粗估参数':>12s}")
print("-" * 44)
for name, d, L in configs:
    p = vocab_size * d + L * 12 * d**2
    print(f"{name:<30s} {p/1e6:>10.1f}M")

&nbsp;

---

## 3.3 注意力配置:GQA 与 head_dim

minimind 的注意力层有三个关键配置:

```python
num_attention_heads = 8      # Q 头数
num_key_value_heads = 4      # KV 头数(比 Q 少 → GQA)
head_dim = 768 // 8 = 96     # 每个头的维度
```

### 3.3.1 GQA(Grouped-Query Attention)

标准多头注意力(MHA)中,Q/K/V 的头数相同。而 minimind 用的是 **GQA**:`num_key_value_heads = 4`(KV 头数只有 Q 头数的一半)。

这意味着 8 个 Q 头**共享** 4 组 KV:每 2 个 Q 头复用同一对 Key/Value。

```
MHA (num_kv_heads = 8):    每个Q头有独立的KV
    Q1→KV1  Q2→KV2  Q3→KV3  Q4→KV4  Q5→KV5  Q6→KV6  Q7→KV7  Q8→KV8

GQA (num_kv_heads = 4):    每2个Q头共享一组KV
    Q1↘         Q3↘         Q5↘         Q7↘
       KV1         KV2         KV3         KV4
    Q2↗         Q4↗         Q6↗         Q8↗
```

GQA 的核心收益:**减少 KV cache 的参数和显存占用**。在自回归生成中,每生成一个 token 都要把之前的 K/V 缓存起来。KV 头数减半,KV cache 的大小也减半。

> 这也是 Llama 2/3、Qwen2/3、DeepSeek-V2 等现代模型的标准选择。当 `num_key_value_heads = 1` 时退化为 **MQA**(Multi-Query Attention),极端省显存但质量略降。GQA 是 MHA 和 MQA 的折中。

### 3.3.2 head_dim = 96

`head_dim = hidden_size // num_attention_heads = 768 // 8 = 96`。

注意:这个值是从 `hidden_size` 和 `num_attention_heads` 自动推导的(代码第 24 行),不是独立配置。

为什么是 96 而不是更常见的 64?因为 `768 / 8 = 96`。这使得每个 Q 头的维度更大,理论上每个头能编码更丰富的信息。

In [ ]:
hidden_size = 768
num_heads = 8
num_kv_heads = 4
head_dim = hidden_size // num_heads  # 96

# Q/KV 投影矩阵的尺寸
q_out = num_heads * head_dim       # 8 × 96 = 768
kv_out = num_kv_heads * head_dim   # 4 × 96 = 384

print(f"head_dim = {hidden_size} // {num_heads} = {head_dim}")
print(f"q_proj:  {hidden_size} × {q_out} = {hidden_size * q_out:,} params")
print(f"k_proj:  {hidden_size} × {kv_out} = {hidden_size * kv_out:,} params")
print(f"v_proj:  {hidden_size} × {kv_out} = {hidden_size * kv_out:,} params")
print(f"o_proj:  {q_out} × {hidden_size} = {q_out * hidden_size:,} params")
print()

# GQA vs MHA 对比
print("=== GQA vs MHA 对比 ===")
print()

# MHA (num_kv_heads = 8)
mha_k_params = hidden_size * (num_heads * head_dim)
mha_v_params = hidden_size * (num_heads * head_dim)
mha_kv_total = mha_k_params + mha_v_params

# GQA (num_kv_heads = 4)
gqa_k_params = hidden_size * (num_kv_heads * head_dim)
gqa_v_params = hidden_size * (num_kv_heads * head_dim)
gqa_kv_total = gqa_k_params + gqa_v_params

print(f"MHA (kv_heads=8): k_proj+v_proj = {mha_kv_total:>10,} params/层")
print(f"GQA (kv_heads=4): k_proj+v_proj = {gqa_kv_total:>10,} params/层")
print(f"GQA 省 KV 投影参数: {1 - gqa_kv_total/mha_kv_total:.0%}")
print()
print(f"q_proj 参数(两者相同): {hidden_size * q_out:>10,}")
print(f"→ GQA 不影响 q_proj 参数量,只缩减 k_proj 和 v_proj")
print()

# KV cache 对比(推理时每 token 缓存的元素数)
# 每层每 token 缓存:2 × num_kv_heads × head_dim (K 和 V)
mha_cache = 2 * num_heads * head_dim
gqa_cache = 2 * num_kv_heads * head_dim
print(f"KV cache 每 token 每层 (MHA): {mha_cache} elements")
print(f"KV cache 每 token 每层 (GQA): {gqa_cache} elements")
print(f"KV cache 节省: {1 - gqa_cache/mha_cache:.0%}")
print(f"8 层 × 32768 tokens 时节省: {(mha_cache - gqa_cache) * 8 * 32768 * 2 / 1e6:.1f}M float16 元素 ≈ {(mha_cache - gqa_cache) * 8 * 32768 * 2 * 2 / 1e9:.2f} GB")

> **小结**:GQA 是 minimind 最聪明的设计之一。仅用 2:1 的 KV 共享比例(4 个 KV 头 vs 8 个 Q 头),就省下了 **50% 的 KV cache 空间**和 **15% 的注意力参数**,而模型质量几乎没有损失。第 4 章会实现 `repeat_kv` 函数,看 GQA 在代码层面如何展开共享的 KV。

&nbsp;

---

## 3.4 FFN 配置:intermediate_size 与 π 的由来

Transformer 的 FeedForward 层(FFN)是参数量的大头。minimind 用的是 **SwiGLU**(SiLU 激活 + GLU 门控),需要三个线性层:`gate_proj`、`up_proj`、`down_proj`。

```python
intermediate_size = math.ceil(hidden_size * math.pi / 64) * 64
# = math.ceil(768 × 3.14159... / 64) × 64
# = math.ceil(37.699...) × 64
# = 38 × 64
# = 2432
```

### 为什么是 π?

这里有一个优雅的设计:SwiGLU 有三个矩阵(gate/up/down),而标准 FFN(如 GPT-2)只有两个。为了保持 FFN 的总参数量大致与传统 $4 \times d^2$ 相当,需要:

$$3 \times d \times d_{\text{ff}} \approx 4 \times d^2 \quad \Longrightarrow \quad d_{\text{ff}} \approx \frac{4}{3} d$$

但 minimind(跟随 Llama / Qwen)取的是更宽的比例:

$$d_{\text{ff}} = \left\lceil \frac{d \times \pi}{64} \right\rceil \times 64 \approx 3.14 \times \frac{d}{2}$$

对于 $d = 768$:$d_{\text{ff}} = 2432 \approx 3.17 \times 768$。

**`× 64` 的对齐**:最后乘以 64 并取整,是为了让 intermediate_size 是 64 的倍数 —— 这是 GPU 张量核心(Tensor Core)高效矩阵乘法的要求。

### hidden_act = 'silu'

SiLU(Sigmoid Linear Unit,$\text{silu}(x) = x \cdot \sigma(x)$)是 SwiGLU 的激活函数。代码里写成 `hidden_act='silu'`,然后通过 `ACT2FN['silu']` 获取对应的 PyTorch 函数。

In [ ]:
hidden_size = 768
intermediate_size = math.ceil(hidden_size * math.pi / 64) * 64

print(f"intermediate_size 的计算过程:")
print(f"  hidden_size × π = {hidden_size} × {math.pi:.6f} = {hidden_size * math.pi:.4f}")
print(f"  ÷ 64 = {hidden_size * math.pi / 64:.4f}")
print(f"  ceil() = {math.ceil(hidden_size * math.pi / 64)}")
print(f"  × 64 = {intermediate_size}")
print()

ratio = intermediate_size / hidden_size
print(f"intermediate / hidden = {intermediate_size} / {hidden_size} = {ratio:.4f}")
print(f"(≈ π/2 = {math.pi/2:.4f},即 SwiGLU 宽度约为 d 的 3.14/2 ≈ 1.57 倍)")
print()

# FFN 三个矩阵的参数量
gate_params = hidden_size * intermediate_size
up_params = hidden_size * intermediate_size
down_params = intermediate_size * hidden_size
ffn_total = gate_params + up_params + down_params

print(f"FFN 参数(每层):")
print(f"  gate_proj: {hidden_size} × {intermediate_size} = {gate_params:>12,}")
print(f"  up_proj:   {hidden_size} × {intermediate_size} = {up_params:>12,}")
print(f"  down_proj: {intermediate_size} × {hidden_size} = {down_params:>12,}")
print(f"  total:                                 {ffn_total:>12,}")
print()

# 与标准 FFN(2 矩阵, 4d 宽度)对比
std_ffn = 2 * hidden_size * (4 * hidden_size)
print(f"标准 FFN (2 × {hidden_size} × {4*hidden_size}):  {std_ffn:>12,}")
print(f"SwiGLU FFN (3 × {hidden_size} × {intermediate_size}):       {ffn_total:>12,}")
print(f"比值: SwiGLU/标准 = {ffn_total/std_ffn:.2f} (接近 1.0,说明参数预算相当)")

> **小结**:SwiGLU 的 FFN 宽度 $d_{\text{ff}} = 2432$ 不是随意选的。它来自 $\lceil d \times \pi / 64 \rceil \times 64$ 的公式,在保持参数预算与标准 FFN 相当的前提下,用门控激活换取更强的表达能力。三个矩阵(gate/up/down)让 FFN 占了每层参数的 **76%** —— FFN 是 minimind 参数量的大头。

&nbsp;

---

## 3.5 位置编码:RoPE 的 base 与 YaRN 外推

Transformer 的注意力机制本身没有位置感知能力。minimind 用 **RoPE**(Rotary Position Embedding,旋转位置编码)来注入位置信息。

### rope_theta = 1e6

`rope_theta`(也叫 `rope_base`)控制旋转的频率分布:

$$\theta_i = \text{base}^{-2i/d}, \quad i = 0, 1, \ldots, d/2-1$$

- `base` 越大 → 高频分量衰减更快 → 模型更依赖低频(全局)位置信息。
- `base` 越小 → 高频分量更强 → 局部位置区分度更高。

minimind 选 `1e6`(而非 Llama 的 `1e4` 或 Qwen3 的 `1e6`),原因是**训练序列短但希望外推长**。高 base 让 RoPE 在更长序列上仍有区分度。

### max_position_embeddings = 32768

这个值定义了模型支持的最大上下文长度。minimind 训练时通常只看 512-2048 token 的序列,但配置里写了 32768,这是为**YaRN 外推**预留的空间。

### rope_scaling(YaRN 配置)

当 `inference_rope_scaling = True` 时,config 会激活一段 YaRN(Yet another RoPE extensioN)配置:

```python
self.rope_scaling = {
    "beta_fast": 32,
    "beta_slow": 1,
    "factor": 16,
    "original_max_position_embeddings": 2048,
    "attention_factor": 1.0,
    "type": "yarn"
}
```

- `original_max_position_embeddings = 2048`:训练时的最大长度。
- `factor = 16`:外推倍数,2048 × 16 = 32768。
- `beta_fast / beta_slow`:频率维度的插值斜坡参数,控制哪些频率做插值、哪些做外推。

YaRN 的核心思想:对 RoPE 的不同频率分量混合使用**插值**(缩放频率)和**外推**(保持不变),在不需要继续训练的情况下扩展上下文长度。第 4 章会推导具体公式。

In [ ]:
# 可视化 rope_theta 对频率分布的影响
import math

head_dim = 96
d_half = head_dim // 2  # 48 个频率分量

for base_name, base in [("Llama (1e4)", 1e4), ("minimind/Qwen3 (1e6)", 1e6)]:
    freqs = [1.0 / (base ** (2*i / head_dim)) for i in range(d_half)]
    min_freq = freqs[-1]  # 最低频(对应最大位置周期)
    max_freq = freqs[0]   # 最高频
    # 最大可区分的位置周期 ≈ 2π / min_freq
    max_period = 2 * math.pi / min_freq
    print(f"rope_theta = {base_name}:")
    print(f"  最高频率: {max_freq:.6f} (周期 ≈ {2*math.pi/max_freq:.1f} tokens)")
    print(f"  最低频率: {min_freq:.2e} (周期 ≈ {max_period:.0f} tokens)")
    print()

# YaRN 外推:训练 2048,推理 32768
train_len = 2048
factor = 16
infer_len = train_len * factor
print(f"YaRN 外推: 训练 {train_len} → 推理 {infer_len} (factor={factor})")
print(f"  原始 max_position_embeddings: {train_len}")
print(f"  外推后: {infer_len}")

> **小结**:`rope_theta=1e6` 配合 `rope_scaling` 的 YaRN 配置(factor=16),让 minimind 能在训练 2048 token 的基础上外推到 32768 token。这是一个「训练省、推理长」的实用策略。第 4 章会逐行推导 `precompute_freqs_cis` 函数,看 RoPE 频率如何计算和 YaRN 如何混合插值/外推。

&nbsp;

---

## 3.6 词表与绑定:vocab_size 与 tie_word_embeddings

### vocab_size = 6400

第 2 章已详细讨论了分词器。这里只看参数量影响:`embed_tokens` 的大小 = `vocab_size × hidden_size = 6400 × 768`。

### tie_word_embeddings = True

这是 minimind 的一个重要设计:**输入嵌入层(`embed_tokens`)和输出层(`lm_head`)共享同一套权重**。

- `embed_tokens`:把 token id 映射成 hidden_size 维向量(查表)。
- `lm_head`:把 hidden_size 维向量映射回 vocab_size 维 logits(线性层)。

如果不绑定,两个矩阵各自独立,参数量翻倍。绑定后:

$$P_{\text{save}} = \text{vocab\_size} \times \text{hidden\_size} = 6400 \times 768 = 4{,}915{,}200$$

省下的 **4.9M 参数**约占总参数的 7.7%。

### _tied_weights_keys 机制

在代码第 268 行:

```python
class MiniMindForCausalLM(PreTrainedModel, GenerationMixin):
    _tied_weights_keys = {"lm_head.weight": "model.embed_tokens.weight"}
```

这告诉 HuggingFace transformers:保存/加载模型时,`lm_head.weight` 和 `model.embed_tokens.weight` 是同一个张量,不要重复存储。绑定在 `__init__` 中通过第 274 行实现:

```python
if self.config.tie_word_embeddings:
    self.model.embed_tokens.weight = self.lm_head.weight
```

In [ ]:
vocab_size = 6400
hidden_size = 768

embed_params = vocab_size * hidden_size

print(f"embed_tokens: {vocab_size} × {hidden_size} = {embed_params:,}")
print(f"lm_head (如果不绑定):  {vocab_size} × {hidden_size} = {embed_params:,}")
print(f"绑定后省下:                                       {embed_params:,} params")
print(f"占总参数比例(64M):  {embed_params / 63_912_192:.1%}")
print()

# 对比:大模型不绑定的代价
print("对比不同词表大小:")
for vs in [6400, 32000, 100000, 150000]:
    p = vs * hidden_size
    tied_saving = p / 1e6
    print(f"  vocab={vs:>6d}: 绑定省 {tied_saving:>6.1f}M params ({p/63_912_192:.0%} of 64M)")
print()
print("→ 词表越大,绑定权重省的参数越多")
print("→ 大模型(vocab=15万)如果不绑定,光嵌入层就占 115M 参数")

> **小结**:绑定权重是一个「免费的午餐」—— 不损失任何表达能力(数学上,输入嵌入和输出投影本就是互逆操作),却省下 4.9M 参数(占 64M 的 7.7%)。对小模型来说,7.7% 的参数节省意义重大。这也是 Llama/Qwen 系列小模型的通用做法。

&nbsp;

---

## 3.7 MoE 配置:use_moe

当 `use_moe = True` 时,每层的 FFN 被替换为 **MOEFeedForward**:一个 router 把 token 路由到 `num_experts` 个专家 FFN 中的一个。

```python
# MoE specific configs (MiniMindConfig — model_minimind.py:~45-50 @67f114a)
self.num_experts = 4             # 4 个专家
self.num_experts_per_tok = 1     # 每个 token 只激活 1 个(top-1 路由)
self.moe_intermediate_size = self.intermediate_size  # 每个专家和 dense FFN 一样宽
self.norm_topk_prob = True       # 路由概率归一化
self.router_aux_loss_coef = 5e-4 # 负载均衡 loss 系数
```

### top-1 路由

`num_experts_per_tok = 1` 意味着每个 token 只经过 1 个专家。这是最激进的稀疏化 —— 每次前向传播只激活 1/4 的 FFN 参数。

### router_aux_loss_coef = 5e-4

如果没有任何约束,router 可能会**把所有 token 都路由到同一个专家**(「赢者通吃」)。auxiliary loss 通过惩罚负载不均衡来避免这个问题:

$$L_{\text{aux}} = \alpha \cdot N_{\text{experts}} \cdot \sum_i \bar{f}_i \cdot \bar{P}_i$$

其中 $\bar{f}_i$ 是专家 $i$ 收到的 token 比例,$\bar{P}_i$ 是 router 给专家 $i$ 的平均概率。当所有专家均匀分担时,$L_{\text{aux}}$ 最小。系数 `5e-4` 很小,确保 aux loss 不干扰主任务。

### Dense vs MoE 参数量

| 配置 | 总参数量 | 激活参数量(每 token) |
|---|---|---|
| Dense (`use_moe=False`) | ~64M | ~64M |
| MoE (`use_moe=True, 4 experts`) | ~198M | ~64M |

MoE 模型的总参数是 dense 的 ~3 倍,但每个 token 只激活 1/4 的 FFN,**激活参数量与 dense 相同**。这就是 MoE 的核心价值:用 3 倍的参数容量,换取与 dense 相当的计算成本。第 15 章会详解 MoE 训练。

In [ ]:
# Dense vs MoE 参数量精确计算
hidden_size = 768
num_layers = 8
vocab_size = 6400
num_heads = 8
num_kv_heads = 4
head_dim = 96
intermediate_size = 2432
num_experts = 4

# === 每层共享部分(Attention + Norms) ===
q_proj = hidden_size * (num_heads * head_dim)
k_proj = hidden_size * (num_kv_heads * head_dim)
v_proj = hidden_size * (num_kv_heads * head_dim)
o_proj = (num_heads * head_dim) * hidden_size
q_norm = head_dim
k_norm = head_dim
attn_per_layer = q_proj + k_proj + v_proj + o_proj + q_norm + k_norm

norm_per_layer = 2 * hidden_size  # input_norm + post_attn_norm

# === Dense FFN ===
dense_ffn = 3 * hidden_size * intermediate_size  # gate + up + down
dense_layer = attn_per_layer + norm_per_layer + dense_ffn

# === MoE FFN ===
router = hidden_size * num_experts  # gate (router) layer
moe_ffn = num_experts * dense_ffn + router
moe_layer = attn_per_layer + norm_per_layer + moe_ffn

# === 全局 ===
embed = vocab_size * hidden_size  # tied, counted once
final_norm = hidden_size

dense_total = embed + num_layers * dense_layer + final_norm
moe_total = embed + num_layers * moe_layer + final_norm
moe_active = embed + num_layers * (attn_per_layer + norm_per_layer + dense_ffn) + final_norm

print("=" * 55)
print(f"{'':>25s} {'Dense':>12s} {'MoE':>12s}")
print("=" * 55)
print(f"{'embed_tokens':>25s} {embed:>12,} {embed:>12,}")
print(f"{'attn / layer':>25s} {attn_per_layer:>12,} {attn_per_layer:>12,}")
print(f"{'norms / layer':>25s} {norm_per_layer:>12,} {norm_per_layer:>12,}")
print(f"{'FFN / layer':>25s} {dense_ffn:>12,} {moe_ffn:>12,}")
print(f"{'  (experts)':>25s} {'—':>12s} {num_experts * dense_ffn:>12,}")
print(f"{'  (router)':>25s} {'—':>12s} {router:>12,}")
print(f"{'layer total':>25s} {dense_layer:>12,} {moe_layer:>12,}")
print(f"{'× ' + str(num_layers) + ' layers':>25s} {num_layers*dense_layer:>12,} {num_layers*moe_layer:>12,}")
print(f"{'final norm':>25s} {final_norm:>12,} {final_norm:>12,}")
print("-" * 55)
print(f"{'TOTAL':>25s} {dense_total:>12,} {moe_total:>12,}")
print(f"{'  (M)':>25s} {dense_total/1e6:>11.2f}M {moe_total/1e6:>11.2f}M")
print(f"{'ACTIVE / token (M)':>25s} {dense_total/1e6:>11.2f}M {moe_active/1e6:>11.2f}M")
print("=" * 55)
print(f"\nMoE 总参数 / Dense 总参数 = {moe_total/dense_total:.2f}x")
print(f"MoE 激活参数 / Dense 激活参数 = {moe_active/dense_total:.2f}x (相同!)")

> **小结**:MoE 的本质是「用 3 倍的参数容量,换取相同的计算成本」。minimind 的 MoE 配置(4 专家、top-1 路由)让总参数膨胀到 198M,但每个 token 只激活 64M —— 与 dense 版本完全相同。`router_aux_loss_coef=5e-4` 确保负载均衡,防止「赢者通吃」。第 15 章会详解 MoE 的训练和蒸馏。

&nbsp;

---

## 3.8 参数量计算:64M 是怎么来的

现在把所有部分加起来,验证 minimind dense 模型的精确参数量。

每层 Transformer Block 的参数:

$$P_{\text{layer}} = P_{\text{attn}} + P_{\text{ffn}} + P_{\text{norm}}$$

其中:

$$P_{\text{attn}} = \underbrace{d \cdot n_q \cdot d_h}_{q\_proj} + \underbrace{d \cdot n_{kv} \cdot d_h}_{k\_proj} + \underbrace{d \cdot n_{kv} \cdot d_h}_{v\_proj} + \underbrace{n_q \cdot d_h \cdot d}_{o\_proj} + \underbrace{2 \cdot d_h}_{q\_norm + k\_norm}$$

$$P_{\text{ffn}} = 3 \cdot d \cdot d_{\text{ff}}$$

$$P_{\text{norm}} = 2 \cdot d$$

总参数(绑定权重):

$$P_{\text{total}} = \underbrace{V \cdot d}_{\text{embed (tied)}} + L \cdot P_{\text{layer}} + \underbrace{d}_{\text{final norm}}$$

### 用代码逐项验证

下面用纯 Python(不依赖 PyTorch/transformers)逐模块计算,模拟 `model.parameters()` 的枚举逻辑:

In [ ]:
import math

# === 所有配置值 ===
d = 768           # hidden_size
L = 8             # num_hidden_layers
V = 6400          # vocab_size
n_q = 8           # num_attention_heads
n_kv = 4          # num_key_value_heads
d_h = d // n_q    # head_dim = 96
d_ff = math.ceil(d * math.pi / 64) * 64  # intermediate_size = 2432

# === 逐项计算 ===
P_embed = V * d
print(f"Embedding (tied):")
print(f"  P_embed = V × d = {V} × {d} = {P_embed:,}")
print()

P_q = d * n_q * d_h
P_k = d * n_kv * d_h
P_v = d * n_kv * d_h
P_o = n_q * d_h * d
P_qk_norm = 2 * d_h
P_attn = P_q + P_k + P_v + P_o + P_qk_norm
print(f"Attention per layer:")
print(f"  P_q     = d × n_q × d_h  = {d} × {n_q} × {d_h} = {P_q:,}")
print(f"  P_k     = d × n_kv × d_h = {d} × {n_kv} × {d_h} = {P_k:,}")
print(f"  P_v     = d × n_kv × d_h = {d} × {n_kv} × {d_h} = {P_v:,}")
print(f"  P_o     = n_q × d_h × d  = {n_q} × {d_h} × {d} = {P_o:,}")
print(f"  P_norm  = 2 × d_h        = 2 × {d_h} = {P_qk_norm}")
print(f"  P_attn  = {P_attn:,}")
print()

P_ffn = 3 * d * d_ff
print(f"FFN per layer:")
print(f"  P_ffn = 3 × d × d_ff = 3 × {d} × {d_ff} = {P_ffn:,}")
print()

P_norm = 2 * d
P_layer = P_attn + P_ffn + P_norm
print(f"Norms per layer: P_norm = 2 × d = {P_norm:,}")
print(f"Layer total: P_layer = P_attn + P_ffn + P_norm = {P_layer:,}")
print()

P_final_norm = d
P_total = P_embed + L * P_layer + P_final_norm
print(f"{'='*50}")
print(f"P_total = P_embed + L × P_layer + P_final_norm")
print(f"        = {P_embed:,} + {L} × {P_layer:,} + {P_final_norm:,}")
print(f"        = {P_total:,}")
print(f"        ≈ {P_total/1e6:.2f}M")
print(f"{'='*50}")

### 逐模块枚举验证

上面的公式手算得到了 63,912,192。现在用纯 Python 逐模块枚举(模拟 `model.parameters()` 的逻辑),验证每一项都精确匹配。

> 如果你已安装 transformers 和 minimind 依赖,也可以直接运行:
> ```python
> from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
> model = MiniMindForCausalLM(MiniMindConfig(hidden_size=768, num_hidden_layers=8))
> print(sum(p.numel() for p in model.parameters()))  # 63,912,192
> ```
> 结果完全一致。在 minimind 仓库的 `utils.py` 里有一个 `get_model_params` 函数,做的就是这件事。

In [ ]:
# 逐模块枚举(与 model_minimind.py 的结构一一对应)

d, L, V = 768, 8, 6400
n_q, n_kv, d_h = 8, 4, d // n_q
d_ff = math.ceil(d * math.pi / 64) * 64

# 逐模块枚举(与 model_minimind.py 的结构一一对应)
modules = {}

# embed_tokens (1 个)
modules['embed_tokens'] = V * d

# 每层
per_layer = {}
per_layer['self_attn.q_proj']   = d * (n_q * d_h)
per_layer['self_attn.k_proj']   = d * (n_kv * d_h)
per_layer['self_attn.v_proj']   = d * (n_kv * d_h)
per_layer['self_attn.o_proj']   = (n_q * d_h) * d
per_layer['self_attn.q_norm']   = d_h
per_layer['self_attn.k_norm']   = d_h
per_layer['input_layernorm']    = d
per_layer['post_attn_layernorm'] = d
per_layer['mlp.gate_proj']      = d * d_ff
per_layer['mlp.down_proj']      = d_ff * d
per_layer['mlp.up_proj']        = d * d_ff

modules[f'layers × {L}'] = sum(per_layer.values()) * L

# final norm
modules['final_norm'] = d

# lm_head: tied → 0 extra params
modules['lm_head (tied)'] = 0

total = sum(modules.values())
print(f"{'Module':<30s} {'Params':>14s}")
print("-" * 46)
for name, params in modules.items():
    if params > 0:
        print(f"{name:<30s} {params:>14,}")
print("-" * 46)
print(f"{'TOTAL':<30s} {total:>14,}")
print(f"{'':>30s} {total/1e6:>13.2f}M")
print()
print(f"✓ 与手算结果一致: 63,912,192 ≈ 64M")

&nbsp;

---

## 3.9 与 Qwen3 的对齐

minimind 的架构设计与 Qwen3(Qwen3Config / Qwen3MoeConfig)完全对齐。这一点在 `scripts/convert_model.py:43-71` 中得到了验证。

### 权重直接映射

`convert_model.py` 的核心逻辑是:把 minimind 训练的 `.pth` 权重直接加载进 Qwen3 的模型结构:

```python
# convert_model.py:43-55
common_config = {
    "vocab_size": lm_config.vocab_size,
    "hidden_size": lm_config.hidden_size,
    "intermediate_size": lm_config.intermediate_size,
    "num_hidden_layers": lm_config.num_hidden_layers,
    "num_attention_heads": lm_config.num_attention_heads,
    "num_key_value_heads": lm_config.num_key_value_heads,
    "head_dim": lm_config.hidden_size // lm_config.num_attention_heads,
    "max_position_embeddings": lm_config.max_position_embeddings,
    "rms_norm_eps": lm_config.rms_norm_eps,
    "rope_theta": lm_config.rope_theta,
    "tie_word_embeddings": lm_config.tie_word_embeddings
}

# convert_model.py:56-62
if not lm_config.use_moe:
    qwen_config = Qwen3Config(**common_config, ...)
    qwen_model = Qwen3ForCausalLM(qwen_config)
    qwen_model.load_state_dict(state_dict, strict=True)  # ← 直接加载!
```

**`strict=True`** 意味着 minimind 的 state_dict 和 Qwen3 的 state_dict **逐参数名、逐形状**完全匹配。这不是巧合 —— minimind 的每一层命名和维度都刻意与 Qwen3 保持一致。

### 为什么对齐

对齐的收益是**生态兼容**:转换后的 minimind 模型可以直接用以下工具加载和推理:

- **llama.cpp**:CPU/边缘端推理
- **vLLM**:高吞吐量 GPU 推理
- **Ollama**:本地一键运行
- **HuggingFace transformers**:标准 `from_pretrained` 接口

这意味着 minimind 虽然只有 64M 参数,但可以享受所有为大模型设计的推理基础设施。

In [ ]:
# 展示 minimind config 与 Qwen3 config 的字段映射
mappings = [
    ("hidden_size",              "hidden_size",              "768"),
    ("num_hidden_layers",        "num_hidden_layers",        "8"),
    ("vocab_size",               "vocab_size",               "6400"),
    ("intermediate_size",        "intermediate_size",        "2432"),
    ("num_attention_heads",      "num_attention_heads",      "8"),
    ("num_key_value_heads",      "num_key_value_heads",      "4"),
    ("head_dim",                 "head_dim (=d//n_heads)",   "96"),
    ("max_position_embeddings",  "max_position_embeddings",  "32768"),
    ("rms_norm_eps",             "rms_norm_eps",             "1e-6"),
    ("rope_theta",               "rope_theta",               "1e6"),
    ("tie_word_embeddings",      "tie_word_embeddings",      "True"),
]

print(f"{'MiniMindConfig':<28s} {'Qwen3Config':<28s} {'值':>8s}")
print("=" * 66)
for mm, qwen, val in mappings:
    print(f"{mm:<28s} {qwen:<28s} {val:>8s}")
print()
print("→ 字段名和值完全一致,权重可直接 load_state_dict(strict=True)")
print()
print("兼容的推理框架:")
for fw in ["llama.cpp", "vLLM", "Ollama", "transformers AutoModel"]:
    print(f"  ✓ {fw}")

> **小结**:`strict=True` 的 `load_state_dict` 是最强的对齐证明 —— minimind 的每一个权重名称和形状都与 Qwen3 逐字段匹配。这不是 API 兼容,而是**权重级兼容**。这意味着 minimind 虽然只有 64M,但它不是「玩具」,而是真正可以用工业级推理工具部署的模型。

&nbsp;

---

## Summary and takeaways

本章逐行拆解了 `MiniMindConfig`(~50 行),解释了每个超参数的取值理由:

| 超参数 | 值 | 核心理由 |
|---|---|---|
| `hidden_size` | 768 | 经典小模型维度,避开 dim<512 的表示瓶颈 |
| `num_hidden_layers` | 8 | 6-12 层是小模型的合理深度,8 层匹配参数预算 |
| `num_attention_heads` | 8 | 标准 8 头,head_dim=96 |
| `num_key_value_heads` | 4 | GQA 2:1,KV cache 省 50% |
| `intermediate_size` | 2432 | `⌈768·π/64⌉·64`,SwiGLU 宽度 ≈ 3.17d |
| `rope_theta` | 1e6 | 高 base 适配短训练 + 长外推 |
| `max_position_embeddings` | 32768 | 为 YaRN 外推预留(factor=16) |
| `vocab_size` | 6400 | 极小词表,embedding 占比合理 |
| `tie_word_embeddings` | True | 省一个 6400×768=4.9M 矩阵 |
| `use_moe` | False | 默认 dense;MoE 模式总参数 198M、激活 64M |
| `router_aux_loss_coef` | 5e-4 | MoE 负载均衡,小系数不干扰主 loss |

**关键认知**:

1. **每个超参数都不是随便选的** —— 它们在参数预算(~64M)、表示能力、训练成本(¥3/2小时)和推理效率之间做了精确的工程取舍。
2. **64M = 4.9M(embed) + 8 × 7.37M(层) + 768(norm)** —— 这就是 minimind 的全部参数。
3. **与 Qwen3 完全对齐** —— 不是模仿,是逐字段、逐权重的兼容,让 minimind 能用整个 LLM 推理生态。

> 下一章我们会进入第一个真正的模型组件:**注意力机制**(RMSNorm + RoPE + GQA)。那里才是张量真正开始流动的地方。

- 精简复习版见 [`./config.ipynb`](./config.ipynb)
- 本章习题与解答见 [`./exercise-solutions.ipynb`](./exercise-solutions.ipynb)

下一章:[第 4 章 · 注意力](../ch04/01_main-chapter-code/README.md)